
# Notebook 13 · Cross-sell basket-aware con stock y margen

## Pregunta de negocio
¿Qué producto complementario conviene recomendar sin empujar SKUs con poco stock o mala rentabilidad?

## Qué aprenderás
- Cómo crear reglas de afinidad entre productos.
- Cómo filtrar por stock disponible y margen.
- Cómo pasar de regla de cesta a recomendación activable.

## Outputs
- `nb13_cross_sell_rules.csv`
- `nb13_cross_sell_customer_candidates.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

from collections import Counter

lineas_pedido = tables["lineas_pedido"].copy()
ventas_pos = tables["ventas_pos"].copy()
stock = tables["stock"].copy()
productos = tables["productos"].copy()
fact = tables["fact"].copy()

online_baskets = lineas_pedido.groupby("order_id")["product_id"].apply(lambda s: sorted(set(s.dropna().astype(str)))).tolist()
pos_baskets = ventas_pos.groupby("ticket_id")["product_id"].apply(lambda s: sorted(set(s.dropna().astype(str)))).tolist()
baskets = online_baskets + pos_baskets

item_count = Counter()
pair_count = Counter()

for basket in baskets:
    for item in basket:
        item_count[item] += 1
    for i in range(len(basket)):
        for j in range(i + 1, len(basket)):
            a, b = sorted((basket[i], basket[j]))
            pair_count[(a, b)] += 1

rules = []
n_baskets = len(baskets)

for (a, b), cnt in pair_count.items():
    support = cnt / n_baskets
    lift = support / ((item_count[a] / n_baskets) * (item_count[b] / n_baskets))
    confidence_a_to_b = cnt / item_count[a]
    confidence_b_to_a = cnt / item_count[b]
    if cnt >= 5:
        rules.append({
            "product_a": a,
            "product_b": b,
            "pair_count": cnt,
            "support": support,
            "lift": lift,
            "confidence_a_to_b": confidence_a_to_b,
            "confidence_b_to_a": confidence_b_to_a,
        })

rules = pd.DataFrame(rules).sort_values(["lift", "pair_count"], ascending=[False, False])
rules.head(15)


,product_a,product_b,pair_count,support,lift,confidence_a_to_b,confidence_b_to_a
2,P1062,P1090,6,0.00324,6.193980,0.130435,0.153846
1,P1047,P1055,5,0.00270,5.674020,0.147059,0.104167
3,P1075,P1082,5,0.00270,5.213964,0.135135,0.104167
0,P1001,P1099,5,0.00270,4.323063,0.098039,0.119048


In [3]:

latest_stock = stock.sort_values("fecha_dt").groupby(["store_id", "product_id"]).tail(1).copy()
latest_stock["available_units"] = latest_stock["stock_cierre"].fillna(0) - latest_stock["stock_reservado"].fillna(0)

prod = productos[["product_id", "nombre_producto", "categoria", "subcategoria", "precio_venta_num", "coste_unitario_num"]].copy()
prod["margen_unitario_est"] = prod["precio_venta_num"] - prod["coste_unitario_num"]

stock_agg = (
    latest_stock.groupby("product_id")
                .agg(
                    stores_with_stock=("store_id", "nunique"),
                    units_available=("available_units", "sum"),
                )
                .reset_index()
)

rules_enriched = (
    rules.merge(prod.add_suffix("_a"), left_on="product_a", right_on="product_id_a", how="left")
         .merge(prod.add_suffix("_b"), left_on="product_b", right_on="product_id_b", how="left")
         .merge(stock_agg.add_suffix("_b"), left_on="product_b", right_on="product_id_b", how="left")
)

rules_enriched["stock_ok"] = rules_enriched["units_available_b"].fillna(0) >= 10
rules_enriched["margen_ok"] = rules_enriched["margen_unitario_est_b"].fillna(0) > rules_enriched["margen_unitario_est_b"].median()
rules_enriched["cross_sell_score"] = (
    rules_enriched["lift"] * 0.5
    + rules_enriched["confidence_a_to_b"] * 0.3
    + np.minimum(rules_enriched["units_available_b"].fillna(0) / 50, 1) * 0.1
    + np.minimum(rules_enriched["margen_unitario_est_b"].fillna(0) / 100, 1) * 0.1
)
rules_filtered = rules_enriched.query("stock_ok == True and margen_ok == True").sort_values("cross_sell_score", ascending=False)
rules_filtered[[
    "product_a", "nombre_producto_a", "product_b", "nombre_producto_b",
    "lift", "confidence_a_to_b", "units_available_b", "margen_unitario_est_b", "cross_sell_score"
]].head(20)


,product_a,nombre_producto_a,product_b,nombre_producto_b,lift,confidence_a_to_b,units_available_b,margen_unitario_est_b,cross_sell_score
0,P1062,Sábana Brisa Pack 4,P1090,Armario Sol XL,6.193980,0.130435,56.0,454.48,3.336120
3,P1001,Cuadro Atlas ★,P1099,Figura Eco ★,4.323063,0.098039,249.0,105.52,2.390943


In [4]:

sales = fact[(fact["tipo_movimiento"] == "VENTA") & fact["customer_id"].notna()].copy()
last_product_per_customer = (
    sales.sort_values("fecha_movimiento_dt")
         .groupby("customer_id")
         .tail(1)[["customer_id", "product_id_std", "categoria", "subcategoria", "fecha_movimiento_dt"]]
         .rename(columns={"product_id_std": "anchor_product"})
)

best_rule_per_anchor = (
    rules_filtered.sort_values("cross_sell_score", ascending=False)
                 .drop_duplicates("product_a")
                 .rename(columns={"product_a": "anchor_product"})
)

customer_candidates = (
    last_product_per_customer
        .merge(best_rule_per_anchor[[
            "anchor_product", "product_b", "nombre_producto_b", "cross_sell_score",
            "units_available_b", "margen_unitario_est_b"
        ]], on="anchor_product", how="left")
        .rename(columns={
            "product_b": "recommended_product",
            "nombre_producto_b": "recommended_product_name",
        })
        .sort_values(["cross_sell_score", "fecha_movimiento_dt"], ascending=[False, False])
)

customer_candidates.head(15)


,customer_id,anchor_product,categoria,subcategoria,fecha_movimiento_dt,recommended_product,recommended_product_name,cross_sell_score,units_available_b,margen_unitario_est_b
23,C0014,P1062,Textil hogar,Sábanas,NaT,P1090,Armario Sol XL,3.336120,56.0,454.48
64,C0087,P1062,Textil hogar,Sábanas,NaT,P1090,Armario Sol XL,3.336120,56.0,454.48
124,C0038,P1062,Textil hogar,Sábanas,NaT,P1090,Armario Sol XL,3.336120,56.0,454.48
38,C0068,P1001,Decoración,Cuadros,NaT,P1099,Figura Eco ★,2.390943,249.0,105.52
87,C0034,P1001,Decoración,Cuadros,NaT,P1099,Figura Eco ★,2.390943,249.0,105.52
14,C0130,P1031,Muebles,Camas,2025-12-08 01:55:00,NaN,NaN,NaN,NaN,NaN
13,CL0079,P1009,Textil hogar,Cortinas,2025-11-07 05:42:25,NaN,NaN,NaN,NaN,NaN
12,C0427,P1088,Decoración,Espejos,2025-11-03 12:27:00,NaN,NaN,NaN,NaN,NaN
11,C0962,P1092,Muebles,Armarios,2025-11-01 00:00:00,NaN,NaN,NaN,NaN,NaN
10,C0555,P1013,Iluminación,Lámparas techo,2025-10-06 03:58:00,NaN,NaN,NaN,NaN,NaN



## Lectura de negocio

Una buena recomendación de cross-sell no solo necesita afinidad de cesta. También necesita:

- stock suficiente,
- margen razonable,
- y una lógica de no saturación del cliente.

Por eso se combinan reglas de asociación con criterios de activación comercial.


In [5]:

path_rules = mpu.export_csv(rules_filtered, "nb13_cross_sell_rules.csv")
path_customers = mpu.export_csv(customer_candidates, "nb13_cross_sell_customer_candidates.csv")

print("Guardado:", path_rules)
print("Guardado:", path_customers)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb13_cross_sell_rules.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb13_cross_sell_customer_candidates.csv
